# Piloto v3: entropia nas posições que o STONE conta — arquivo único e completo
Reúne tudo o que foi sendo adicionado (v2 + truncagem + bootstrap + testes pareados + verificação de configuração).

## Como usar
1. Ambiente de execução > Alterar tipo > **T4 GPU**.
2. Edite **só a célula "Configuração"** (MODE, DELTA, N_PROBLEMS...). Depois rode **tudo em ordem**, de cima para baixo.
3. Para trocar `MODE` ou `DELTA` numa nova rodada: rode de novo a partir da célula "Derivar configuração" (não precisa recarregar o modelo).
   Para trocar o modelo (`MODEL_ID`): rode de novo a partir de "Carregar modelo".
4. Cada rodada salva em arquivo próprio no Drive (o nome inclui modo, modelo e delta), então uma rodada não sobrescreve outra.

## O que cada MODE significa
- `repo`: o código oficial do STONE **como foi publicado**: elegibilidade pelo token argmax, `prefix_length=0` (lista verde fixa),
  vocabulário de 50272 ids, correspondência exata para tokens sintáticos, modelo Qwen2.5-Coder-3B-Instruct e amostragem do
  `generation_config` do modelo (o `run.py` não define temperatura nem top-k).
- `paper`: o que o **artigo descreve**: candidato amostrado, hash do token anterior (prefix 1), vocabulário inteiro,
  T=1.0, top-k=50, sem penalidade de repetição, modelo base. (O artigo usa o 7B; aqui o padrão é o 3B base, que cabe na T4.)

Observação: a diferença repo × artigo vem da minha leitura do código e do PDF. Não foi confirmada com os autores.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes evalplus scikit-learn scipy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/stone_pilot'
os.makedirs(OUT_DIR, exist_ok=True)

## Configuração (edite aqui)

In [ ]:
MODE = 'paper'          # 'repo' ou 'paper' (ver explicação no topo)
MODEL_ID = None         # None = modelo padrão do modo; ou escreva um id, ex.: 'Qwen/Qwen2.5-Coder-7B' (com QUANT_4BIT=True)
QUANT_4BIT = False      # True para modelos de 7B na T4
GAMMA = 0.5
DELTA = 0.5             # artigo: 0.5 (HumanEval+) e 1.0 (MBPP+). Use 1.0 / 2.0 como teste de sensibilidade.
HASH_KEY = 15485863
MAX_NEW = 200
MIN_LEN = 200           # só usado no modo repo (como no run.py)
N_PROBLEMS = 50         # 164 = HumanEval+ inteiro
GEN_UNWM = True         # gerar também sem marca (negativo para AUROC e teste pareado)
TAUS = [0.3, 0.6, 0.9, 1.2]
SEED = 0
RESUME = True           # retoma de onde parou se a MESMA configuração já tiver arquivo salvo

PRESETS = {
    'repo': dict(model='Qwen/Qwen2.5-Coder-3B-Instruct', PREFIX=0, ELIG='argmax', SYN='exact', GV=50272,
                 GEN=dict(do_sample=True, min_length=MIN_LEN, no_repeat_ngram_size=4),
                 PROBE_T=None, PROBE_K=None),
    'paper': dict(model='Qwen/Qwen2.5-Coder-3B', PREFIX=1, ELIG='sampled', SYN='broad', GV=None,
                  GEN=dict(do_sample=True, temperature=1.0, top_k=50, top_p=1.0, repetition_penalty=1.0),
                  PROBE_T=1.0, PROBE_K=50),
}

## Carregar modelo

In [ ]:
import torch, math, json, random, functools, collections, re, time
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList

MODEL_ID = MODEL_ID or PRESETS[MODE]['model']
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if QUANT_4BIT:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model.eval()
print('MODEL_ID:', MODEL_ID, '| memória GPU (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))

## Derivar configuração (rode de novo ao trocar MODE ou DELTA)

In [ ]:
P = PRESETS[MODE]
CFG = dict(PREFIX=P['PREFIX'], ELIG=P['ELIG'], SYN=P['SYN'])
V = model.get_output_embeddings().weight.shape[0]          # largura dos logits
GV = min(P['GV'] or V, V)                                  # tamanho do vocabulário usado na lista verde
GEN_KW = dict(P['GEN'], max_new_tokens=MAX_NEW, pad_token_id=tok.eos_token_id)
TAG = f"{MODE}_{MODEL_ID.split('/')[-1]}_d{DELTA}"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

eff = {k: GEN_KW.get(k, getattr(model.generation_config, k, None))
       for k in ['do_sample', 'temperature', 'top_k', 'top_p', 'repetition_penalty']}
print('MODE:', MODE, '| CFG:', CFG, '| GV:', GV, '| V:', V)
print('MODEL_ID:', MODEL_ID, '| 4bit:', QUANT_4BIT, '| GAMMA:', GAMMA, '| DELTA:', DELTA)
print('GEN_KW:', GEN_KW)
print('amostragem efetiva:', eff)
print('TAG da rodada:', TAG)
if MODE == 'paper' and (eff['temperature'] != 1.0 or eff['top_k'] != 50 or eff['repetition_penalty'] != 1.0):
    print('ATENÇÃO: a amostragem efetiva NÃO é a do artigo (T=1.0, top_k=50, sem penalidade).')

## Conjunto sintático, lista verde e entropia

In [ ]:
KEYWORDS = '''True False None and as assert async await break class continue def del elif else
except finally for from global if import in is lambda nonlocal not or pass raise return try while
with yield'''.split()
OPS = ['+', '-', '*', '/', '%', '**', '//', '=', '==', '!=', '>', '<', '>=', '<=', '+=', '-=',
       '*=', '/=', '%=', '//=', '**=', '&', '|', '<<', '>>', '^', '~']
DELIMS = ['(', ')', '[', ']', '{', '}', ',', ':', '.', ';', '@', '->', '...']
WS = [' ', '\t', '\n']
TYPES = 'int float complex str bytes bool list tuple set dict NoneType'.split()
SYN_SET = set(KEYWORDS + OPS + DELIMS + WS + TYPES)
PUNCT_CHARS = set('()[]{},:.;@->=+*/%<>!&|^~')

_syn_cache = {}
def is_syn(tid: int) -> bool:
    """'exact' = como o stone.py (token ou token.strip() na lista). 'broad' = regra ampla minha (v1)."""
    key = (CFG['SYN'], tid)                       # o cache depende do modo, então não fica "velho"
    if key in _syn_cache:
        return _syn_cache[key]
    d = tok.decode([tid], skip_special_tokens=True)
    if CFG['SYN'] == 'exact':
        r = (d in SYN_SET) or (d.strip() in SYN_SET)
    else:
        st = d.strip()
        r = (st == '') or (st in SYN_SET) or all(c in PUNCT_CHARS for c in st)
    _syn_cache[key] = r
    return r

def seed_key(prev_ids):
    """prefix 0 -> chave constante (lista fixa, como no repo). prefix 1 -> id do token anterior % GV."""
    if CFG['PREFIX'] == 0:
        return 1
    return int(prev_ids[-1]) % GV

@functools.lru_cache(maxsize=8192)
def green_mask_cpu(key: int, gv: int, gamma: float, hash_key: int) -> torch.Tensor:
    g = torch.Generator(device='cpu')
    g.manual_seed(hash_key * key)
    perm = torch.randperm(gv, generator=g)
    m = torch.zeros(gv, dtype=torch.bool)
    m[perm[:int(gv * gamma)]] = True
    return m
# Obs.: o repo gera a permutação na GPU; aqui é na CPU, então a lista exata difere, mas as propriedades são as mesmas.

def is_green(key: int, tid: int) -> bool:
    return tid < GV and bool(green_mask_cpu(key, GV, GAMMA, HASH_KEY)[tid])

def entropy_rows(logits):
    logp = torch.log_softmax(logits.float(), dim=-1)
    p = logp.exp()
    return (-torch.where(p > 0, p * logp, torch.zeros_like(p)).sum(-1))

## Geração com o processador do STONE

In [ ]:
class StoneProc(LogitsProcessor):
    """Espelha o processador do STONE. Registra, por passo, a entropia (antes do viés) e se o viés foi aplicado."""
    def __init__(self):
        self.recs = []
    def __call__(self, input_ids, scores):
        key = seed_key(input_ids[0].tolist())
        H = float(entropy_rows(scores[0]))
        if CFG['ELIG'] == 'argmax':                                   # stone.py: argmax(raw_probs)
            probe = int(torch.argmax(scores[0]))
        else:                                                         # Alg. 1 do artigo: candidato amostrado
            vals, idx = torch.topk(scores[0].float() / P['PROBE_T'], P['PROBE_K'])
            probe = int(idx[torch.multinomial(torch.softmax(vals, -1), 1)])
        eligible = not is_syn(probe)
        if eligible:
            gm = torch.zeros(scores.shape[-1], dtype=torch.bool, device=scores.device)
            gm[:GV] = green_mask_cpu(key, GV, GAMMA, HASH_KEY).to(scores.device)
            scores = scores.clone()
            scores[0][gm] += DELTA
        self.recs.append(dict(H=H, eligible=eligible))
        return scores

@torch.no_grad()
def generate(prompt, watermark=True):
    enc = tok(prompt, return_tensors='pt').to(model.device)
    proc = StoneProc() if watermark else None
    lp = LogitsProcessorList([proc]) if watermark else None
    out = model.generate(**enc, logits_processor=lp, **GEN_KW)
    return out[0].tolist(), enc.input_ids.shape[1], (proc.recs if watermark else None)

## Pontuação (teacher forcing) e z-score

In [ ]:
@torch.no_grad()
def score_seq(ids, prompt_len):
    """Linhas (H, verde, sintático, na_completion, token_id) para cada posição a partir de PREFIX."""
    t = torch.tensor([ids], device=model.device)
    Hs = entropy_rows(model(t).logits[0]).cpu().tolist()
    H_at = [-1e4] + Hs[:-1]                       # entropia da distribuição que previu o token idx (como no sweet.py)
    rows = []
    for idx in range(CFG['PREFIX'], len(ids)):
        key = seed_key(ids[:idx])
        tid = ids[idx]
        rows.append((H_at[idx], is_green(key, tid), is_syn(tid), idx >= prompt_len, tid))
    return rows

def zscore(rows, tau=None, excl_syn=True, scope='all'):
    sel = [g for (H, g, syn, inc, _) in rows
           if (scope == 'all' or inc) and (not (excl_syn and syn)) and (tau is None or H > tau)]
    n = len(sel)
    return 0.0 if n == 0 else (sum(sel) - GAMMA * n) / math.sqrt(GAMMA * (1 - GAMMA) * n)

# Métodos comparados. ATENÇÃO: só a DETECÇÃO muda; a geração é sempre a do STONE.
METHODS = {'KGW (todos)': dict(excl_syn=False, tau=None),
           'SWEET t=0.9 (sem excl. sintática)': dict(excl_syn=False, tau=0.9),
           'STONE': dict(excl_syn=True, tau=None)}
for _t in TAUS:
    METHODS[f'STONE + H>{_t}'] = dict(excl_syn=True, tau=_t)

## Rodar no HumanEval+ (salva a cada problema; retoma se a configuração for a mesma)

In [ ]:
from evalplus.data import get_human_eval_plus
probs = get_human_eval_plus()
task_ids = list(probs.keys())[:N_PROBLEMS]

meta = dict(MODE=MODE, MODEL_ID=MODEL_ID, DELTA=DELTA, GAMMA=GAMMA, GV=GV, CFG=CFG, GEN_KW=GEN_KW,
            MAX_NEW=MAX_NEW, HASH_KEY=HASH_KEY, SEED=SEED, GEN_UNWM=GEN_UNWM)
meta = json.loads(json.dumps(meta))
SAVE = f'{OUT_DIR}/he_plus_{TAG}.json'
META = SAVE.replace('.json', '_meta.json')

results = []
if RESUME and os.path.exists(SAVE) and os.path.exists(META):
    if json.load(open(META)) == meta:
        results = json.load(open(SAVE))
        print(f'Retomando: {len(results)} problemas já feitos em {SAVE}')
    else:
        SAVE = SAVE.replace('.json', f'_{int(time.time())}.json')
        META = SAVE.replace('.json', '_meta.json')
        print('Já existe arquivo com configuração DIFERENTE; salvando em', SAVE)
elif os.path.exists(SAVE):
    SAVE = SAVE.replace('.json', f'_{int(time.time())}.json')
    META = SAVE.replace('.json', '_meta.json')
    print('Arquivo existente sem meta ou RESUME=False; salvando em', SAVE)
json.dump(meta, open(META, 'w'))

done = {r['task_id'] for r in results}
for i, tid in enumerate(task_ids):
    if tid in done:
        continue
    prompt = probs[tid]['prompt']
    wm_ids, plen, recs = generate(prompt, True)
    r = dict(task_id=tid, plen=plen, recs=recs, wm_rows=score_seq(wm_ids, plen))
    if GEN_UNWM:
        un_ids, plen_u, _ = generate(prompt, False)
        r['un_rows'] = score_seq(un_ids, plen_u)
    hu_ids = tok(prompt + probs[tid]['canonical_solution']).input_ids
    r['hu_rows'] = score_seq(hu_ids, len(tok(prompt).input_ids))
    results.append(r)
    json.dump(results, open(SAVE, 'w'))
    print(f'{len(results)}/{len(task_ids)} {tid} gen={len(wm_ids) - plen} tokens')

## Análise 1: o que o detector conta e como a entropia se distribui (completions completas, sem truncar)

In [ ]:
import matplotlib.pyplot as plt

gen_H = np.array([x['H'] for r in results for x in r['recs'] if x['eligible']])
print(f'[geração] posições com viés aplicado: {len(gen_H)} de {sum(len(r["recs"]) for r in results)}')

cnt = [row for r in results for row in r['wm_rows'] if row[3] and not row[2]]
H_c = np.array([row[0] for row in cnt]); G_c = np.array([row[1] for row in cnt]); T_c = [row[4] for row in cnt]
print(f'\n[detector] tokens contados na completion: {len(H_c)}')
print(f'Entropia: média={H_c.mean():.3f} mediana={np.median(H_c):.3f} desvio={H_c.std():.3f}')
print('Quantis 10/25/50/75/90:', np.round(np.quantile(H_c, [.1, .25, .5, .75, .9]), 3))
for t in TAUS:
    print(f'  H <= {t}: {np.mean(H_c <= t):.1%}')
print(f'Tokens contados com id >= {GV} (nunca verdes): {np.mean(np.array(T_c) >= GV):.1%}')
print(f'\nProporção de verdes geral: {G_c.mean():.3f} (gamma={GAMMA})')
bins = [-1, 0.3, 0.6, 0.9, 1.2, 2, 100]
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (H_c > lo) & (H_c <= hi)
    if m.sum():
        print(f'  H em ({lo},{hi}]: n={m.sum():5d}  verdes={G_c[m].mean():.3f}')

low = collections.Counter(repr(tok.decode([t])) for t, h in zip(T_c, H_c) if h <= 0.3)
print('\nTokens contados com H<=0.3 mais frequentes:', low.most_common(20))

rec_all = [(x['eligible'], row) for r in results for x, row in zip(r['recs'], [rw for rw in r['wm_rows'] if rw[3]])]
el = [row for e, row in rec_all if e]; nel = [row for e, row in rec_all if not e]
if el:  print(f'\nViés aplicado, mas token final sintático (não contado): {np.mean([row[2] for row in el]):.1%}')
if nel: print(f'Viés NÃO aplicado, mas token final contado pelo detector: {np.mean([not row[2] for row in nel]):.1%}')

plt.figure(figsize=(7, 4))
plt.hist(H_c, bins=50)
for t in TAUS: plt.axvline(t, ls='--', c='gray')
plt.xlabel('Entropia (tokens contados pelo STONE)'); plt.ylabel('Contagem'); plt.title(TAG)
plt.savefig(f'{OUT_DIR}/hist_{TAG}.png', dpi=150, bbox_inches='tight'); plt.show()

# Ver algumas completions
for r in results[:3]:
    ids = [row[4] for row in r['wm_rows'] if row[3]]
    print('=' * 20, r['task_id'], f'({len(ids)} tokens)')
    print(tok.decode(ids)[:1200])

## Truncar completions no fim da função (sem regenerar)
Corta o que vem depois da função (testes, doctest, prosa). Como o modelo é causal, a entropia e a cor dos tokens anteriores ao corte
não mudam, então a truncagem pós-hoc é exata para esses tokens.

In [ ]:
STOP_RE = re.compile(r'\n(?=[^\s])|```|<\|endoftext\|>|<\|im_end\|>')   # volta à coluna 0, bloco markdown ou EOS

def truncate_rows(rows):
    prompt_part = [row for row in rows if not row[3]]
    comp = [row for row in rows if row[3]]
    text, cut_k = '', len(comp)
    for k, row in enumerate(comp):
        text += tok.decode([row[4]])
        if STOP_RE.search(text):
            cut_k = k            # descarta o token que revelou o fim da função
            break
    return prompt_part + comp[:cut_k]

results_trunc = []
for r in results:
    r2 = dict(r)
    for k in ['wm_rows', 'un_rows', 'hu_rows']:
        if k in r:
            r2[k] = truncate_rows(r[k])
    results_trunc.append(r2)

n_before = np.mean([sum(1 for row in r['wm_rows'] if row[3]) for r in results])
n_after = np.mean([sum(1 for row in r['wm_rows'] if row[3]) for r in results_trunc])
print(f'tokens de completion por amostra marcada: {n_before:.0f} -> {n_after:.0f}')

## Análise 2: z médio, AUROC com IC (bootstrap sobre problemas)
Leitura: o z médio de texto não marcado e de código humano deveria ficar perto de 0 (calibração). O escopo `all` (prompt+completion)
é dominado pelo prompt, que é igual nos três textos; para a pergunta "a marca funciona?", olhe o escopo `completion` e o negativo não marcado.

In [ ]:
from scipy.stats import rankdata, wilcoxon

def fast_auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    n1, n2 = len(pos), len(neg)
    r = rankdata(np.concatenate([pos, neg]))
    return (r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n2)

def zvec(res, key, scope, kw):
    return np.array([zscore(r[key], scope=scope, **kw) for r in res])

NEGS = [('humano', 'hu_rows')] + ([('não-marcado', 'un_rows')] if GEN_UNWM else [])

def evaluate(res, label, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    print(f'\n########## {label} ##########')
    for scope in ['completion', 'all']:
        print(f'\n=== escopo: {"só completion" if scope == "completion" else "prompt+completion"} (n={len(res)} problemas) ===')
        for name, kw in METHODS.items():
            pos = zvec(res, 'wm_rows', scope, kw)
            line = f'{name:34s} marcado z={pos.mean():+.2f}'
            for nlabel, nkey in NEGS:
                neg = zvec(res, nkey, scope, kw)
                n = len(pos)
                a = fast_auc(pos, neg)
                bs = [fast_auc(pos[i], neg[i]) for i in (rng.integers(0, n, n) for _ in range(B))]
                lo, hi = np.percentile(bs, [2.5, 97.5])
                line += f' | {nlabel}: z={neg.mean():+.2f} AUROC={a:.3f} [{lo:.2f},{hi:.2f}]'
            print(line)

evaluate(results, 'completions completas')
evaluate(results_trunc, 'completions truncadas')

## Análise 3: comparação direta entre métodos (pareada por problema)
- Diferença de AUROC em relação ao STONE, com IC por bootstrap pareado.
- Δz = z(marcado) − z(não marcado) por problema (mesmo prompt), com Wilcoxon; e Δ(Δz) de cada método contra o STONE.
Cuidado: são vários testes sem correção para múltiplos testes; um p ou IC limítrofe pode ser acaso.

In [ ]:
def paired_boot(res, m1, m2, neg_key, scope, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    p1, n1 = zvec(res, 'wm_rows', scope, METHODS[m1]), zvec(res, neg_key, scope, METHODS[m1])
    p2, n2 = zvec(res, 'wm_rows', scope, METHODS[m2]), zvec(res, neg_key, scope, METHODS[m2])
    n = len(res)
    d = np.array([fast_auc(p2[i], n2[i]) - fast_auc(p1[i], n1[i]) for i in (rng.integers(0, n, n) for _ in range(B))])
    return d.mean(), np.percentile(d, [2.5, 97.5])

def compare_auc(res, label, scopes=('completion',)):
    print(f'\n##### {label}: diferença de AUROC em relação ao STONE #####')
    for scope in scopes:
        for nlabel, nkey in NEGS:
            print(f'\n[{scope} | negativo: {nlabel}]')
            for m in METHODS:
                if m == 'STONE':
                    continue
                mean, (lo, hi) = paired_boot(res, 'STONE', m, nkey, scope)
                flag = '  <- IC exclui 0' if (lo > 0 or hi < 0) else ''
                print(f'  {m:34s} {mean:+.3f} [{lo:+.3f}, {hi:+.3f}]{flag}')

def safe_wilcoxon(d):
    try:
        return wilcoxon(d).pvalue
    except ValueError:
        return float('nan')

def compare_dz(res, label):
    if not GEN_UNWM:
        print('GEN_UNWM=False: sem texto não marcado, pulando Δz.')
        return
    print(f'\n##### {label}: Δz = z(marcado) − z(não marcado), só completion #####')
    dzs = {}
    for name, kw in METHODS.items():
        d = zvec(res, 'wm_rows', 'completion', kw) - zvec(res, 'un_rows', 'completion', kw)
        dzs[name] = d
        sd = d.std(ddof=1)
        print(f'{name:34s} Δz médio={d.mean():+.2f}  d pareado={d.mean() / sd if sd > 0 else float("nan"):+.2f}  p(Wilcoxon)={safe_wilcoxon(d):.3f}')
    print('\nEfeito de cada método menos o efeito do STONE (por problema)')
    for name in METHODS:
        if name == 'STONE':
            continue
        d = dzs[name] - dzs['STONE']
        print(f'{name:34s} Δ(Δz)={d.mean():+.2f}  p(Wilcoxon)={safe_wilcoxon(d):.3f}')

compare_auc(results, 'completions completas')
compare_auc(results_trunc, 'completions truncadas')
compare_dz(results_trunc, 'completions truncadas')
compare_dz(results, 'completions completas')

## O que NÃO concluir daqui
- Só a detecção muda entre os métodos da tabela; a geração é sempre a do STONE. "STONE + H>τ" ainda não é a combinação completa
  (que aplicaria o viés só onde H>τ e mudaria a geração e a qualidade).
- Se o STONE sozinho não se distingue do texto não marcado (Δz perto de 0), a marca está fraca nessa configuração e comparar métodos
  não é informativo. Aumente DELTA (sensibilidade) ou revise a amostragem antes de comparar.
- Resultados de uma configuração (modelo, δ, T, top-k) não se transferem para outra sem testar.
- Qualidade do código (pass@1) não foi medida neste notebook.